In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

## Graphical Representation

In [ ]:
import { display } from "tslab";
import { Graphviz } from "@hpcc-js/wasm";
const gv = Graphviz.load();

The function `toDot` takes four arguments:
- `A` is an array of natural numbers of length $n$,
- `f` is a natural number such that $0 \leq f < n$ holds,
- `g` is a natural number such that $f < g < n$ holds,
- `u` is a natural number such that $0 \leq u < n$ holds.
  This argument is optional.

The function returns a graphical representation of the array `A` as a heap. 
This graphical representation is stored as a directed graph with an encoding suitable for `graphviz`. 

The part `A[0:g]` is represented as a binary tree, while the part `A[g:]` is represented
as an array.  Furthermore, all indexes in the range `A[k:g]` satisfy the heap condition.  The nodes in the range `[0:k-1]`
are colored red.  If `u` is set, the node `A[u]` is colored orange.

In [ ]:
type HeapEntry<T> = [number, T & { mIndex: number }];

class Digraph {
  private lines: string[] = [];
  constructor() {
    this.lines.push('digraph {');
    this.lines.push('  node [shape=record, style="rounded"];');
  }
  node(id: string, opts: { label?: string; style?: string; color?: string } = {}) {
    const attrs: string[] = [];
    if (opts.label) attrs.push(`label="${opts.label}"`);
    if (opts.style) attrs.push(`style="${opts.style}"`);
    if (opts.color) attrs.push(`color="${opts.color}"`);
    this.lines.push(`  ${id} [${attrs.join(", ")}];`);
  }
  edge(a: string, b: string) { this.lines.push(`  ${a} -> ${b};`); }
  toString() { return this.lines.concat(["}"]).join("\n"); }
}

In [ ]:
function heapToDot<T>(A: HeapEntry<T>[], k = 0, g?: number, u?: number): string {
  const n = A.length;
  const dot = new Digraph();

  for (let i = 0; i < n; i++) {
    const [p, o] = A[i];
    const s = String(o);
    const idx = String(o.mIndex ?? "");
    const label = `{ ${p} | ${s} | ${idx} }`;

    let color: string | undefined;
    if (i < k) color = "red";
    if (typeof u === "number" && i === u) color = "orange";
    dot.node(String(i), { label, color });
  }

  for (let i = 0; i < Math.floor(n / 2); i++) {
    const l = 2 * i + 1, r = 2 * i + 2;
    if (l < n) dot.edge(String(i), String(l));
    if (r < n) dot.edge(String(i), String(r));
  }
  return dot.toString();
}

# Priority Queues implemented as Heaps

## Auxiliary Methods

The function call `swap(A, i, j)` takes an array `A` and  two indexes `i` and `j` and exchanges the elements at these indexes.

In [ ]:
function swap<T>(A: HeapEntry<T>[], i: number, j: number): void {
  const [pi, oi] = A[i];
  const [pj, oj] = A[j];
  A[i] = [pj, oj];
  A[j] = [pi, oi];
  oi.mIndex = j;
  oj.mIndex = i;
}

The function `ascend` takes two arguments:
- `A` is an array
- `k` is an index into the array `A`.

   Therefore we have $k \in \bigl\{0, \cdots, \texttt{len}(A)-1\bigr\}$.

The array `A` represents a *heap*.  However, the <em style="color:blue">heap condition</em> might be violated 
at index `k`: It might be the case that the element at this index is to small and needs to rise to the top
of the heap.  The function `ascend` will fix the heap condition and will rise the element `A[k]` as much 
as is necessary to turn `A` into a heap.

In [ ]:
function ascend<T>(A: HeapEntry<T>[], k: number): number {
  while (k > 0) {
    const p = Math.floor((k - 1) / 2);
    if (A[k][0] < A[p][0]) {
      swap(A, k, p);
      k = p;
    } else {
      return k;
    }
  }
  return k;
}

The function `descend(A)` takes one argument `A` where `A` is an array that is organized as a heap,
but possibly has its heap condition violated at its root, i.e. at index `0`.  The
purpose of the procedure `descend` is to restore the heap condition at the root.
We initialize a variable `k` as `0` and the `while`-loop proceeds as follows: 
- We compute the index `j` of the left subtree below index `k`.
- We check whether there also is a right subtree at position `j+1`.
  
  This is the case if `j + 1 <= n` where `n = len(A) - 1`.  
- If the heap condition is violated at index `k`, we exchange the element at  position `k` 
  with the child that has the higher priority, i.e. the child that is smaller. 
- Next, we check in line 9 whether the heap condition is violated at index `k`.  
  If the heap condition is satisfied, there is nothing left to do and the procedure returns.  
  
- Otherwise, the element at position `k` is swapped with
  the element at position `j`.  
  
  Of course, after this swap it is possible that the heap condition is
  violated at position `j`.  Therefore,  `k` is set to `j` and the `while`-loop continues
  as long as the node at position `k` has at least one child, i.e. as long as 
  `2 * k + 1 <= n`.

In [ ]:
function descend<T>(A: HeapEntry<T>[]): void {
  const n = A.length;
  let k = 0;
  while (2 * k + 1 < n) {
    let j = 2 * k + 1;
    if (j + 1 < n && A[j][0] > A[j + 1][0]) j = j + 1;
    if (A[k][0] <= A[j][0]) return;
    swap(A, k, j);
    k = j;
  }
}

## Implementing the API

The function `insert(H, x)` takes two arguments:
- `H` is a heap that is represented as an array.
- `x` is a pair of the form `(p, o)` where
  - `p` is a natural number interpreted as a priority.  The smaller the number, the higher the priority.
  - `o` is an object.  
  
    Every object `o` knows its index in the heap via the member variable `o.mIndex`.
    
This method inserts the pair `x` into the heap `H`.  Furthermore, the object `o` is modified so that it remembers
the index at which it is stored in `H`.  This is done by storing this index in `o.mIndex`.

In [ ]:
function insert<T>(H: HeapEntry<T>[], x: HeapEntry<T>): number {
  const n = H.length;
  H.push(x);
  x[1].mIndex = n;
  const k = ascend(H, n);
  x[1].mIndex = k;
  return k;
}


The function `elevate(H, o, p)` takes three arguments.
- `H` is an array that is organized as a heap.
- `o` is an object that occurs in the heap `H` at index `o.mIndex`, i.e. we have `H[o.mIndex] = p_old, o.mIndex`,
  where `p_old` is the priority that was used when `o` was stored in `H`.
- `p` is the new priority of `o` in `H`.  This priority must be higher than the priority `p_old`, i.e. we must have `p < p_old`.

The function call `elevate(H, o, p)` elevates the priority of the object `o` to `p` in `H` and takes care that `o` is stored further up in `H` 
so that the heap property of `H` is maintained.

In [ ]:
function elevate<T>(H: HeapEntry<T>[], o: T & { mIndex: number }, p: number): void {
  const k = o.mIndex;
  H[k] = [p, o] as HeapEntry<T>;
  ascend(H, k);
}

In [ ]:
function remove<T>(H: HeapEntry<T>[]): HeapEntry<T> | undefined {
  if (H.length === 0) return undefined;
  const first = H[0];
  const last = H.pop()!;
  first[1].mIndex = -1;

  if (H.length === 0) return first;

  H[0] = last;
  last[1].mIndex = 0;
  descend(H);
  return first;
}

## Testing

In [ ]:
class Node {
  constructor(public mValue: unknown, public mIndex: number = -1) {}

  toString(): string { return String(this.mValue); }

  repr(): string {
    const idx = this.mIndex !== undefined ? `:${this.mIndex}` : "";
    return `Node(${this.mValue}${idx})`;
  }

  equals(other: Node | null): boolean {
    if (other === null) return false;
    return this.mValue === other.mValue;
  }

  lessThan(other: Node): boolean {
    return (this.mValue as any) < (other.mValue as any);
  }
}

In [ ]:
async function demo1() {
  const letters = Array.from({ length: 26 }, (_, i) => String.fromCharCode(97 + i));
  const L: [number, Node][] = letters.map((c, i) => [i, new Node(c)]);

  const H: [number, Node][] = [];
  const graphviz = await gv;

  for (const x of L) {
    insert(H, x);
  }

  const svg = graphviz.layout(heapToDot(H, 0, undefined, undefined), "svg", "dot");
display.html(svg);

  console.log('Elevating "w" to priority 2:');
  const w = H.find(([, o]) => String(o.mValue) === "w")?.[1] ?? L[22][1];
  elevate(H, w, 2);
  const svg2 = graphviz.layout(heapToDot(H), "svg", "dot");
  display.html(svg);
}

In [ ]:
demo1();

In [ ]:
function randomInt(minIncl: number, maxIncl: number): number {
  return Math.floor(Math.random() * (maxIncl - minIncl + 1)) + minIncl;
}

In [ ]:
async function heap_sort<T>(L: [number, T & { mIndex: number }][]): Promise<number[]> {
  const H: Array<[number, any]> = [];
  const graphviz = await gv;

  for (const x of L) {
    insert(H, x);
    const svg = graphviz.layout(heapToDot(H), "svg", "dot");
    display.html(svg);
  }

  const S: number[] = [];
  while (H.length > 0) {
    const p = remove(H);
    const svg = graphviz.layout(heapToDot(H), "svg", "dot");
    display.html(svg);
    S.push(p[0]);
  }
  return S;
}

In [ ]:
async function demo2() {
  const Lraw = Array.from({ length: 12 }, () => randomInt(1, 200));
  const L: [number, Node][] = Lraw.map((n) => [n, new Node(n)]);
  console.log("L =", L);
  const S = await heap_sort(L);
  console.log("S =", S);
}

In [ ]:
demo2();

In [ ]:
// Hey Tobi, hier ist meine eigene Implementierung von der Übersetzung.
// Ich hatte keine Ahnung, wie man das sonst übersetzten soll.
function heapToDotIndex<T>(A: HeapEntry<T>[]): string {
  const n = A.length;
  const dot = new Digraph();

  for (let k = 0; k < n; k++) {
    const [p, o] = A[k];
    const label = `{ ${p} | ${(o as any).mIndex} }`;
    dot.node(String(k), { label, style: "rounded" });
  }

  for (let k = 0; k < Math.floor(n / 2); k++) {
    const l = 2 * k + 1;
    const r = 2 * k + 2;
    if (l < n) dot.edge(String(k), String(l));
    if (r < n) dot.edge(String(k), String(r));
  }

  return dot.toString();
}


In [ ]:
async function demo2() {
  const Lraw = Array.from({ length: 12 }, () => randomInt(1, 200));
  const L: [number, Node][] = Lraw.map((n) => [n, new Node(n)]);
  console.log("L =", L);

  // Heap aufbauen, aber NICHT anzeigen:
  const H: HeapEntry<Node>[] = [];
  for (const x of L) insert(H, x);

  const graphviz = await gv;
  const S: number[] = [];

  // Nur ABBau zeigen:
  while (H.length > 0) {
    const p = remove(H);
    const svg = graphviz.layout(heapToDotIndex(H), "svg", "dot");
    display.html(svg);
    S.push(p[0]);
  }

  console.log("S =", S);
}


In [ ]:
demo2();